In [ ]:
import os
import numpy as np
import pandas as pd
from tifffile import imwrite
from scipy.spatial import KDTree

def compute_knn_features(points, k=5):
    if len(points) == 0:
        return [], [], []

    setX = points['cx'] / points['cx'].max()
    setY = points['cy'] / points['cy'].max()
    setZ = points['cz'] / points['cz'].max()

    coords = np.stack([setX, setY, setZ], axis=1)

    tree = KDTree(coords)
    k_use = min(k, len(coords))
    distances, indices = tree.query(coords, k=k_use)

    distAv = []
    pWT = []
    pC = []

    for i in range(len(indices)):
        neighbor_indices = indices[i]
        neighbor_phenotypes = points.iloc[neighbor_indices]['phenotype']

        count_WT = sum(p.startswith('WT') for p in neighbor_phenotypes)
        count_C = sum(p.startswith('C') for p in neighbor_phenotypes)
        total = len(neighbor_phenotypes)

        distAv.append(np.median(distances[i]))
        pWT.append(count_WT)
        pC.append(count_C)

    return distAv, pWT, pC

In [ ]:
pathF = r"E:\organoids_paper\2025\frames-nuclei\cellpose\with_channels\files_new\OSCAR_channels_wt025_c05_rel2"
dataFolder = pd.read_csv(os.path.join(pathF, "summary_results.txt"), sep='\t')
pattern = r'^(?P<series_id>.+?)-frame-(?P<frame>\d+)\.txt$'

# Aplicar el regex
dataFolder[['series_id', 'frame']] = dataFolder['file'].str.extract(pattern)

# Convertir frame a entero (soporta NaNs)
dataFolder['frame'] = dataFolder['frame'].astype(pd.Int64Dtype())


# print(dataFolder['series_id'].unique())

# TARGET_SERIES = "TL09"  # Cambia esto por el ID que necesites
# # Seleccionar solo el target
# dataFolder= dataFolder[dataFolder['series_id'] == TARGET_SERIES]

# spacing_x = 0.325  # micrones por pixel en X
# spacing_y = 0.325  # micrones por pixel en Y
# spacing_z = 1.0    # micrones entre planos

In [ ]:
##this is a dirty trick if you just want to insepct a specific series_id
# TARGET_SERIES = "TL09"  # Cambia esto por el ID que necesites
# Seleccionar solo el target
# dataFolder= dataFolder[dataFolder['series_id'] == TARGET_SERIES]
###
output_path = os.path.join(pathF, "distances")
os.makedirs(output_path, exist_ok=True)

# Resultados generales
summary_stats = []

for sid in dataFolder['series_id'].unique():
    subset = dataFolder[dataFolder['series_id'] == sid].sort_values(by='frame')
    databyTL = pd.DataFrame()  # DataFrame acumulativo por serie_id

    for _, row in subset.iterrows():
        fileN = "Summary_" + row['file']
        series_id = row['series_id']
        frame = row['frame']
        filePath = os.path.join(pathF, fileN)

        try:
            oscarDATA = pd.read_csv(filePath, sep='\t')
        except Exception as e:
            print(f"❌ Error leyendo {fileN}: {e}")
            continue
        
        ## i check if the spatial cooridnates x,y,z (named as cx, cy,cz by OSCAR in my case) are present, but you can change the names and verify the columns you need
        if not all(col in oscarDATA.columns for col in ['cx', 'cy', 'cz', 'phenotype']):
            print(f"⚠️ Columnas faltantes en {fileN}")
            continue

        oscarDATA['frame'] = frame
        oscarDATA['series_id'] = series_id

        # Filtrar por fenotipos
        totalCells = len(oscarDATA)
        pointsWT = oscarDATA[oscarDATA['phenotype'].str.startswith("WT")].copy()
        pointsC = oscarDATA[oscarDATA['phenotype'].str.startswith("C")].copy()

        # Calcular KNN
        distAv, pWT, pC = compute_knn_features(oscarDATA)
        oscarDATA['K-NN(k=5)'] = distAv
        oscarDATA['Num_WT'] = pWT
        oscarDATA['Num_C'] = pC

        # Guardar estadísticas del archivo actual
        summary_stats.append({
            'file': fileN,
            'series_id': series_id,
            'frame': frame,
            'total_cells': totalCells,
            'WT_cells': len(pointsWT),
            'C_cells': len(pointsC),
            'median_knn_dist': np.median(distAv) if len(distAv) > 0 else np.nan,
            'mean_Num_WT': np.mean(pWT) if len(pWT) > 0 else np.nan,
            'mean_Num_C': np.mean(pC) if len(pC) > 0 else np.nan
        })

        databyTL = pd.concat([databyTL, oscarDATA], ignore_index=True)

    # Guardar datos por serie_id
    temp_path = os.path.join(output_path, f"{sid}_df.txt")
    databyTL.to_csv(temp_path, sep='\t', index=False)   
        
    # #
    if sid == TARGET_SERIES:
        max_frame =   int(databyTL['frame'].max()) + 1
        max_z = int(  databyTL['cz'].max()) + 1
        max_y = int(  databyTL['cy'].max()) + 1
        max_x = int(  databyTL['cx'].max()) + 1
        #
        array_5d = np.zeros((max_frame, 3, max_z, max_y, max_x), dtype=np.uint8)
        for tt in range(max_frame):
            for _, row in databyTL.iterrows():
                x, y, z = int(row['cx']), int(row['cy']), int(row['cz'])
                frame = int(row['frame'])
                phenotype = row['phenotype']
                array_5d[frame, 0, z, y, x] = 255  # Total
                if phenotype.startswith("WT"):
                    array_5d[frame, 1, z, y, x] = 255
                elif phenotype.startswith("C"):
                    array_5d[frame, 2, z, y, x] = 255
    
        # Reorganizar ejes a (T, Z, C, Y, X)
        array_5d = np.transpose(array_5d, (0, 2, 1, 3, 4))

        # Guardar como TIFF con metadatos para ImageJ
        tiff_path = os.path.join(output_path, f"{sid}_stack_TZCYX.tiff")
        imwrite(
            tiff_path,
            array_5d,
            photometric='minisblack',
            metadata={
                'axes': 'TZCYX',  # Especificar orden de ejes
                'spacing': 1,     # Opcional: ajustar según calibración real
                'unit': 'um'      # Opcional: unidades físicas
            },
            imagej=True
        )
        print(f"✅ TIFF guardado correctamente: {tiff_path}")

# Guardar resumen total
summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(os.path.join(output_path, "summary_statistics_KNN.txt"), sep='\t', index=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Asegurar tipo correcto
summary_df['frame'] = summary_df['frame'].astype(int)

# Agregar %WT
summary_df['pct_WT'] = summary_df['WT_cells'] / summary_df['total_cells'] * 100

# Calcular medias por frame
mean_df = summary_df.groupby('frame').agg({
    'median_knn_dist': 'mean',
    'pct_WT': 'mean'
}).reset_index()

# --- PLOT Median KNN ---
plt.figure(figsize=(10, 5))

# Líneas por series_id
sns.lineplot(data=summary_df, x='frame', y='median_knn_dist', hue='series_id',
             palette='tab10', linewidth=1, alpha=0.5, legend=False)

# Dots por frame y TL
sns.stripplot(data=summary_df, x='frame', y='median_knn_dist', hue='series_id',
              dodge=True, jitter=False, alpha=0.5, size=6, palette='tab10', legend=False)


plt.title("Median KNN distance per frame")
plt.xlabel("Frame")
plt.ylabel("Median KNN distance")
plt.legend(title='TL (series_id)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# --- PLOT % WT ---
plt.figure(figsize=(10, 5))

# Líneas por series_id
sns.lineplot(data=summary_df, x='frame', y='pct_WT', hue='series_id',
             palette='tab10', linewidth=1, alpha=0.5, legend=False)

# Dots por frame y TL
sns.stripplot(data=summary_df, x='frame', y='pct_WT', hue='series_id',
              dodge=True, jitter=False, alpha=0.5, size=6, palette='tab10', legend=False)

plt.title("% WT cells per frame")
plt.xlabel("Frame")
plt.ylabel("Percentage of WT cells")
plt.legend(title='TL (series_id)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
